<a href="https://colab.research.google.com/github/pablonvsx/pisi3-ufrpe/blob/main/data-science/notebooks/ML/primeira_analise/light_gbm/Experimento_light_gbm_(3_classes_com_pesos_manuais).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Preparação do ambiente


In [ ]:
# INSTALAÇÃO DO LIGHTGBM
!pip install lightgbm -q

In [ ]:
# IMPORTAÇÃO DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

In [ ]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# CONFIGURAÇÃO DE CAMINHO E CARREGAMENTO
if IN_COLAB:
    print("Ambiente Google Colab detectado.")

    drive.mount('/content/drive')

    DATA_PATH = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/amostra_rotulada_3.parquet"
    )

else:
    print("Ambiente local/VS Code detectado.")

    DATA_PATH = Path("../../dataset/amostra_rotulada_3.parquet")

# LEITURA DO DATASET
df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente Google Colab detectado.
Mounted at /content/drive
Dataset Parquet carregado com sucesso.
Shape do dataset: (141399, 22)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,...,93.116725,Good,1,1,1,1,3.7,1,5,Adequada
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Adequada
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,...,100.000000,Excellent,1,1,1,1,3.7,1,5,Adequada
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Adequada
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,...,92.882835,Good,1,1,1,1,2.0,1,5,Adequada


In [ ]:
# PREPARANDO O AMBIENTE PARA MACHINE LEARNING
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
# DEFINIÇÃO DE X E Y
X = df[
    [
        "Temperature (cel)",
        "Orthophosphate (mg/l)",
        "Country",
        "Waterbody Type",
        "Nitrogen (mg/l)"
    ]
]

y = df["conama_status"]

In [ ]:
# TRAIN/TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

print("\nDistribuição das classes no treino:")
print(y_train.value_counts(normalize=True).round(4))

print("\nDistribuição das classes no teste:")
print(y_test.value_counts(normalize=True).round(4))

Treino: (113119, 5)
Teste: (28280, 5)

Distribuição das classes no treino:
conama_status
Adequada    0.7318
Atenção     0.2585
Crítica     0.0098
Name: proportion, dtype: float64

Distribuição das classes no teste:
conama_status
Adequada    0.7318
Atenção     0.2585
Crítica     0.0098
Name: proportion, dtype: float64


In [ ]:
# PRÉ-PROCESSAMENTO
categorical_features = [
    "Country",
    "Waterbody Type"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [ ]:
class_weights = {
    "Adequada": 1,
    "Atenção": 2,
    "Crítica": 10
}

In [ ]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LGBMClassifier(
                random_state=SEED,
                class_weight=class_weights,
                n_jobs=-1,
                verbose=-1
            )
        )
    ]
)

In [ ]:
# TREINAMENTO
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type'])])),
                ('classifier',
                 LGBMClassifier(class_weight={'Adequada': 1, 'Atenção': 2,
                                              'Crítica': 10},
                                n_jobs=-1, random_state=42, verbose=-1))])

## Avaliação das Métricas de Treino



In [ ]:
y_train_pred = model.predict(X_train)

In [ ]:
train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_recall = recall_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_confusion_matrix = confusion_matrix(
    y_train,
    y_train_pred
)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("\nClassification Report:")
print(classification_report(y_train, y_train_pred))

print("Train Confusion Matrix:")
print(train_confusion_matrix)

Train Accuracy:
0.8062394469540926
Train Precision:
0.8209081256905917
Train Recall:
0.8062394469540926
Train F1:
0.811958123958309

Classification Report:
              precision    recall  f1-score   support

    Adequada       0.90      0.85      0.87     82775
     Atenção       0.62      0.71      0.66     29238
     Crítica       0.28      0.46      0.34      1106

    accuracy                           0.81    113119
   macro avg       0.60      0.67      0.63    113119
weighted avg       0.82      0.81      0.81    113119

Train Confusion Matrix:
[[70060 12289   426]
 [ 7707 20636   895]
 [   33   568   505]]


## Avaliação das Métricas de Teste


In [ ]:
y_pred = model.predict(X_test)

In [ ]:
# MÉTRICAS DE TESTE — LGBM 5 variáveis com balanceamento com pesos manuais
test_accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:")
print(test_accuracy)

print("\nDiferença treino/teste (overfitting):")
print(round(train_accuracy - test_accuracy, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.7997524752475248

Diferença treino/teste (overfitting):
0.0065

Classification Report:
              precision    recall  f1-score   support

    Adequada       0.90      0.84      0.87     20694
     Atenção       0.61      0.69      0.65      7309
     Crítica       0.17      0.29      0.21       277

    accuracy                           0.80     28280
   macro avg       0.56      0.61      0.58     28280
weighted avg       0.82      0.80      0.81     28280


Confusion Matrix:
[[17478  3101   115]
 [ 1974  5059   276]
 [   10   187    80]]


In [ ]:
erros = X_test[
    (y_test == "Crítica") &
    (y_pred == "Atenção")
]

acertos = X_test[
    (y_test == "Crítica") &
    (y_pred == "Crítica")
]

In [ ]:
erros.describe()


,Temperature (cel),Orthophosphate (mg/l),Nitrogen (mg/l)
count,187.000000,187.000000,187.000000
mean,13.085080,3.775904,17.216081
std,4.448811,3.697154,10.580538
min,2.300000,0.020000,0.200000
25%,11.460000,0.458000,9.960000
50%,11.460000,3.090000,16.500000
75%,15.000000,6.440000,24.250000
max,30.000000,25.900000,45.500000


In [ ]:

acertos.describe()

,Temperature (cel),Orthophosphate (mg/l),Nitrogen (mg/l)
count,80.000000,80.000000,80.000000
mean,11.784000,6.254937,15.148575
std,4.695504,8.325052,3.849873
min,2.400000,0.020000,9.620000
25%,11.460000,2.805000,12.300000
50%,11.460000,5.080000,14.350000
75%,11.460000,6.757500,17.350000
max,37.100000,64.600000,31.000000


In [ ]:
analise_erros = X_test.copy()

analise_erros["Classe_Real"] = y_test
analise_erros["Classe_Predita"] = y_pred

In [ ]:
critica_para_atencao = analise_erros[
    (analise_erros["Classe_Real"] == "Crítica") &
    (analise_erros["Classe_Predita"] == "Atenção")
]

In [ ]:
print(critica_para_atencao.shape)

(187, 7)


In [ ]:
critica_para_atencao.sample(20, random_state=42)

,Temperature (cel),Orthophosphate (mg/l),Country,Waterbody Type,Nitrogen (mg/l),Classe_Real,Classe_Predita
93814,11.46,0.144,England,Effluent,5.0,Crítica,Atenção
43796,16.90,8.310,England,Effluent,14.7,Crítica,Atenção
71056,12.20,2.130,England,Effluent,13.3,Crítica,Atenção
108897,11.46,0.144,England,Effluent,5.0,Crítica,Atenção
10587,11.46,8.250,England,Effluent,28.4,Crítica,Atenção
71187,14.30,2.070,England,River,11.2,Crítica,Atenção
41776,20.01,3.900,England,Effluent,13.5,Crítica,Atenção
76773,11.46,0.088,England,Effluent,40.3,Crítica,Atenção
106047,8.70,3.480,England,River,12.0,Crítica,Atenção
103015,21.30,6.430,England,Effluent,30.9,Crítica,Atenção


In [ ]:
critica_correta = analise_erros[
    (analise_erros["Classe_Real"] == "Crítica") &
    (analise_erros["Classe_Predita"] == "Crítica")
]

In [ ]:
print("CRÍTICAS ACERTADAS")
print(critica_correta.describe())

print("\nCRÍTICAS CONFUNDIDAS COM ATENÇÃO")
print(critica_para_atencao.describe())

CRÍTICAS ACERTADAS
       Temperature (cel)  Orthophosphate (mg/l)  Nitrogen (mg/l)
count          80.000000              80.000000        80.000000
mean           11.784000               6.254937        15.148575
std             4.695504               8.325052         3.849873
min             2.400000               0.020000         9.620000
25%            11.460000               2.805000        12.300000
50%            11.460000               5.080000        14.350000
75%            11.460000               6.757500        17.350000
max            37.100000              64.600000        31.000000

CRÍTICAS CONFUNDIDAS COM ATENÇÃO
       Temperature (cel)  Orthophosphate (mg/l)  Nitrogen (mg/l)
count         187.000000             187.000000       187.000000
mean           13.085080               3.775904        17.216081
std             4.448811               3.697154        10.580538
min             2.300000               0.020000         0.200000
25%            11.460000             

## Análise dos testes utilizando pesos manuais

> *nota atualizada

Após observar o comportamento excessivamente agressivo do parâmetro `class_weight='balanced'`, foram realizados experimentos utilizando pesos definidos manualmente para cada classe.

O objetivo desses testes foi encontrar um ponto de equilíbrio entre dois cenários extremos:

* Sem balanceamento: o modelo praticamente ignorava a classe Crítica.
* Balanceamento automático: o modelo passava a classificar uma quantidade excessiva de amostras como Crítica.

Foram avaliadas diferentes combinações de pesos:

### Teste 1

```python
Adequada = 1
Atenção = 3
Crítica = 10
```

Nesse cenário, o modelo apresentou melhora na detecção da classe Crítica em relação à versão sem balanceamento. Entretanto, observou-se que o peso atribuído à classe Atenção ainda era elevado, fazendo com que uma grande quantidade de amostras críticas fosse direcionada para essa classe.

A matriz de confusão mostrou que a maioria dos erros da classe Crítica continuava sendo convertida em Atenção, indicando que ambas as classes apresentam características muito semelhantes nas variáveis disponíveis.

### Teste 2

```python
Adequada = 1
Atenção = 2
Crítica = 10
```

Esse experimento apresentou o melhor equilíbrio entre as métricas avaliadas.

A redução do peso da classe Atenção permitiu que mais amostras críticas fossem corretamente identificadas sem provocar um crescimento exagerado dos falsos positivos.

Os resultados mostraram:

* Acurácia próxima de 80%.
* Boa capacidade de generalização.
* Melhor equilíbrio entre precisão e recall da classe Crítica.
* Melhor F1-score para a classe Crítica dentre os testes realizados.

Por esse motivo, este foi considerado o melhor conjunto de pesos encontrado até o momento.

### Teste 3

```python
Adequada = 1
Atenção = 2
Crítica = 12
```

Neste experimento foi aumentado o peso da classe Crítica na tentativa de melhorar ainda mais sua capacidade de detecção.

De fato, o recall da classe Crítica aumentou. Porém, a precisão diminuiu consideravelmente.

Esse comportamento reproduziu parcialmente o que já havia sido observado com o `class_weight='balanced'`: o modelo passou a classificar mais amostras como Crítica, mas grande parte dessas classificações não era correta.

Em vez de aprender melhor os padrões da classe Crítica, o modelo apenas deslocou novamente sua fronteira de decisão em direção a essa classe.

### Conclusões dos testes

Os experimentos mostraram que ajustes de peso influenciam diretamente o equilíbrio entre precisão e recall.

Quando o peso da classe Crítica é muito baixo, o modelo tende a ignorá-la.

Quando o peso é excessivamente alto, o modelo passa a prever Crítica em excesso, gerando muitos falsos positivos.

O melhor resultado obtido foi com:

```python
Adequada = 1
Atenção = 2
Crítica = 10
```

No entanto, mesmo nesse cenário, a principal fonte de erro continuou sendo a confusão entre as classes **Crítica** e **Atenção**.

Esse comportamento sugere que o problema não está apenas no balanceamento, mas possivelmente na própria estrutura dos dados, na sobreposição entre classes e na distribuição temporal das amostras, hipóteses que serão investigadas nas próximas etapas do estudo.
